In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

YEARS = list(range(2018, 2025))
COUNTRIES = ["Switzerland", "United States", "Germany", "South Africa", "China", "India"]
MIN_BASE = 20   # previous year applications below this make the % change unstable (team to confirm)

apps = pd.read_csv("clean-pt2/patents_yearly_applications_6countries.csv")
gran = pd.read_csv("clean-pt2/patents_yearly_granted_6countries.csv")

apps = apps[(apps["field"] == "All") & apps["year"].isin(YEARS)]
gran = gran[(gran["field"] == "All") & gran["year"].isin(YEARS)]

# Full 6 x 7 grid so any missing country-year shows up as NaN, not as a missing row
grid = pd.MultiIndex.from_product([COUNTRIES, YEARS], names=["country", "year"])
panel = (pd.DataFrame(index=grid).reset_index()
         .merge(apps[["country", "year", "num_patent_applications", "complete"]]
                .rename(columns={"complete": "apps_complete"}), on=["country", "year"], how="left")
         .merge(gran[["country", "year", "num_patent_granted", "complete"]]
                .rename(columns={"complete": "grants_complete"}), on=["country", "year"], how="left"))
panel["group"] = np.where(panel["country"].isin(["Switzerland", "United States", "Germany"]),
                          "Developed", "Developing")
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

# Quality measure and transform
panel["grant_rate"] = (panel["num_patent_granted"] / panel["num_patent_applications"]
                       ).where(panel["num_patent_applications"] > 0)
panel["log_apps"] = np.log1p(panel["num_patent_applications"])

# Year-on-year % change, blanked where the previous year is too small to be reliable
prev = panel.groupby("country")["num_patent_applications"].shift(1)
panel["apps_yoy_pct"] = (panel["num_patent_applications"] / prev - 1) * 100
panel["apps_yoy_unstable"] = prev < MIN_BASE
panel.loc[panel["apps_yoy_unstable"], "apps_yoy_pct"] = np.nan

# CAGR 2018 to 2024 (6 growth periods), descriptive only
first = panel[panel["year"] == 2018].set_index("country")["num_patent_applications"]
last = panel[panel["year"] == 2024].set_index("country")["num_patent_applications"]
cagr = ((last / first) ** (1 / 6) - 1) * 100

print(panel.shape, "rows (expect 42)")
print(panel.isna().sum()[lambda s: s > 0])
print("\nYear-on-year % change in applications (blank = unstable or first year)")
print(panel.pivot(index="country", columns="year", values="apps_yoy_pct").round(1))
print("\nCAGR 2018 to 2024 (%)")
print(cagr.round(1))

(42, 11) rows (expect 42)
num_patent_granted     4
grants_complete        4
grant_rate             4
apps_yoy_pct          20
dtype: int64

Year-on-year % change in applications (blank = unstable or first year)
year           2018  2019  2020  2021   2022  2023  2024
country                                                 
China           NaN  60.3  49.4  29.3   15.6  25.9  30.7
Germany         NaN  34.1  20.2  -1.0   17.5   9.0 -35.3
India           NaN -95.6   NaN   NaN    NaN   NaN   NaN
South Africa    NaN   NaN   NaN   NaN  208.7  39.4  35.4
Switzerland     NaN   NaN   NaN   NaN    NaN   NaN   NaN
United States   NaN  12.8   2.4  -7.6  -22.4 -33.6 -34.5

CAGR 2018 to 2024 (%)
country
China            34.4
Germany           4.7
India           -55.8
South Africa     63.6
Switzerland      -4.7
United States   -15.6
Name: num_patent_applications, dtype: float64


#### Population data and calculates patents per 100,000 people

In [2]:
print(list(Path(".").rglob("*POP*")))

[WindowsPath('clean-pt2/population_6countries.csv'), WindowsPath('.venv/Lib/site-packages/pandas/tests/series/methods/test_pop.py'), WindowsPath('.venv/Lib/site-packages/pandas/tests/frame/methods/test_pop.py'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stdlib/poplib.pyi'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stubs/pexpect/pexpect/popen_spawn.pyi'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stdlib/multiprocessing/popen_fork.pyi'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stdlib/multiprocessing/popen_forkserver.pyi'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stdlib/multiprocessing/popen_spawn_posix.pyi'), WindowsPath('.venv/Lib/site-packages/jedi/third_party/typeshed/stdlib/multiprocessing/popen_spawn_win32.pyi')]


In [3]:
import json
from urllib.request import urlopen

ISO = {"Switzerland": "CHE", "United States": "USA", "Germany": "DEU",
       "South Africa": "ZAF", "China": "CHN", "India": "IND"}

# 1. Download total population from the World Bank API
url = ("https://api.worldbank.org/v2/country/CHE;USA;DEU;ZAF;CHN;IND/"
       "indicator/SP.POP.TOTL?date=2018:2024&format=json&per_page=100")
with urlopen(url) as r:
    records = json.load(r)[1]

pop = pd.DataFrame([{"country": d["country"]["value"], "iso3": d["countryiso3code"],
                     "year": int(d["date"]), "population": d["value"]} for d in records])
pop["country"] = pop["iso3"].map({v: k for k, v in ISO.items()})
pop = pop[["country", "year", "population"]]
pop.to_csv("clean-pt2/population_6countries.csv", index=False)

# 2. Merge into the patents panel and calculate patents per 100,000 people
panel = panel.drop(columns=["population"], errors="ignore").merge(pop, on=["country", "year"], how="left")
panel["apps_per_100k"] = panel["num_patent_applications"] / panel["population"] * 100_000
panel["grants_per_100k"] = panel["num_patent_granted"] / panel["population"] * 100_000

print("Rows in population table:", len(pop), "(expect 42)")
print("Missing population values:", panel["population"].isna().sum())
print("\nApplications per 100,000 people")
print(panel.pivot(index="country", columns="year", values="apps_per_100k").round(3))

Rows in population table: 42 (expect 42)
Missing population values: 0

Applications per 100,000 people
year            2018   2019   2020   2021   2022   2023    2024
country                                                        
China          2.140  3.418  5.094  6.581  7.607  9.590  12.548
Germany        0.888  1.188  1.426  1.411  1.658  1.805   1.164
India          0.020  0.001  0.001  0.000  0.000  0.001   0.000
South Africa   0.012  0.008  0.010  0.037  0.114  0.157   0.209
Switzerland    0.047  0.035  0.058  0.126  0.068  0.090   0.033
United States  4.456  5.001  5.101  4.706  3.633  2.393   1.552


#### Min-Max scaling to 0 to 100

In [4]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min()) * 100

panel["apps_scaled_linear"] = minmax(panel["apps_per_100k"])
panel["apps_scaled_log"] = minmax(np.log1p(panel["apps_per_100k"]))

print("Linear scaling (0 to 100)")
print(panel.pivot(index="country", columns="year", values="apps_scaled_linear").round(1))
print("\nLog scaling (0 to 100)")
print(panel.pivot(index="country", columns="year", values="apps_scaled_log").round(1))

panel.to_csv("clean-pt2/patents_panel_features.csv", index=False)

Linear scaling (0 to 100)
year           2018  2019  2020  2021  2022  2023   2024
country                                                 
China          17.1  27.2  40.6  52.4  60.6  76.4  100.0
Germany         7.1   9.5  11.4  11.2  13.2  14.4    9.3
India           0.2   0.0   0.0   0.0   0.0   0.0    0.0
South Africa    0.1   0.1   0.1   0.3   0.9   1.2    1.7
Switzerland     0.4   0.3   0.5   1.0   0.5   0.7    0.3
United States  35.5  39.9  40.6  37.5  29.0  19.1   12.4

Log scaling (0 to 100)
year           2018  2019  2020  2021  2022  2023   2024
country                                                 
China          43.9  57.0  69.3  77.7  82.6  90.5  100.0
Germany        24.4  30.0  34.0  33.8  37.5  39.6   29.6
India           0.7   0.0   0.0   0.0   0.0   0.0    0.0
South Africa    0.5   0.3   0.4   1.4   4.1   5.6    7.3
Switzerland     1.8   1.3   2.2   4.6   2.5   3.3    1.3
United States  65.1  68.8  69.4  66.8  58.8  46.9   35.9


In [5]:
import numpy as np
import pandas as pd


def calculate_research_sub_index():
  # 1. Create a sample/mock DataFrame representing your raw data (2018-2024 for 6 countries)
  # Replace this with your actual imported dataset (e.g., pd.read_csv('your_data.csv'))
  np.random.seed(42)
  countries = [
      "Switzerland",
      "United States",
      "Germany",
      "South Africa",
      "China",
      "India",
  ]
  years = list(range(2018, 2025))

  data = []
  for country in countries:
    base_pop = (
        8.7
        if country == "Switzerland"
        else (330
              if country == "United States"
              else (83
                    if country == "Germany"
                    else (60 if country == "South Africa" else 1400))
        )
    )
    for year in years:
      pop = base_pop * 1_000_000
      pubs = np.random.randint(500, 50000) * (1.05 ** (year - 2018))
      patents = np.random.randint(100, 20000) * (1.08 ** (year - 2018))
      citation_impact = np.random.uniform(0.8, 2.5)

      data.append({
          "Country": country,
          "Year": year,
          "Population": pop,
          "AI_Publications": pubs,
          "AI_Patents": patents,
          "Citation_Impact": citation_impact,
      })

  df = pd.DataFrame(data)

  # 2. Per Capita Normalization (per 100,000 population)
  df["Pubs_Per_100k"] = (df["AI_Publications"] / df["Population"]) * 100000
  df["Patents_Per_100k"] = (df["AI_Patents"] / df["Population"]) * 100000

  # 3. Quality Weighting (Publications and Patents weighted by Citation Impact)
  df["Weighted_Pubs"] = df["Pubs_Per_100k"] * df["Citation_Impact"]
  df["Weighted_Patents"] = df["Patents_Per_100k"] * df["Citation_Impact"]


  # 4. Min-Max Scaling function to map indicators to a 0-100 scale
  def min_max_scale(series):
    min_val = series.min()
    max_val = series.max()
    if max_val - min_val == 0:
      return pd.Series(0, index=series.index)
    return ((series - min_val) / (max_val - min_val)) * 100


  df["Norm_Weighted_Pubs"] = min_max_scale(df["Weighted_Pubs"])
  df["Norm_Weighted_Patents"] = min_max_scale(df["Weighted_Patents"])

  # 5. Aggregate into the Research Sub-Index
  df["Research_Sub_Index"] = (
      df["Norm_Weighted_Pubs"] + df["Norm_Weighted_Patents"]
  ) / 2

  output_cols = [
      "Country",
      "Year",
      "Pubs_Per_100k",
      "Patents_Per_100k",
      "Research_Sub_Index",
  ]
  print(df[output_cols].head(14))

  return df


# Execute the script
research_df = calculate_research_sub_index()

          Country  Year  Pubs_Per_100k  Patents_Per_100k  Research_Sub_Index
0     Switzerland  2018     187.298851         11.034483           10.564889
1     Switzerland  2019     545.903448        141.318621           42.585632
2     Switzerland  2020     219.866379         60.679614           26.115474
3     Switzerland  2021     593.861625        233.452145          100.000000
4     Switzerland  2022      17.729626         39.610558            5.793119
5     Switzerland  2023     582.219111        298.391821           78.019769
6     Switzerland  2024     449.671403        118.486616           50.886076
7   United States  2018      12.707273          5.770303            1.243288
8   United States  2019       1.033136          0.094582            0.042495
9   United States  2020      12.118480          0.706556            1.017825
10  United States  2021      11.243345          6.875737            1.104302
11  United States  2022       1.616992          3.684864            0.517872

In [6]:
# View rows for a specific country (e.g., South Africa)
print(research_df[research_df["Country"] == "South Africa"])

# Or view a random sample across all countries
print(research_df.sample(10))

         Country  Year  Population  AI_Publications    AI_Patents  \
21  South Africa  2018  60000000.0     27057.000000   1095.000000   
22  South Africa  2019  60000000.0     12635.700000   8347.320000   
23  South Africa  2020  60000000.0      1671.390000  10064.865600   
24  South Africa  2021  60000000.0     25202.653875  15475.561920   
25  South Africa  2022  60000000.0     48132.831994  11796.799772   
26  South Africa  2023  60000000.0     49192.996545  27250.158512   
27  South Africa  2024  60000000.0     66313.292681  12863.203262   

    Citation_Impact  Pubs_Per_100k  Patents_Per_100k  Weighted_Pubs  \
21         1.816430      45.095000          1.825000      81.911909   
22         1.133171      21.059500         13.912200      23.864012   
23         1.717477       2.785650         16.774776       4.784289   
24         2.440934      42.004423         25.792603     102.530025   
25         1.269199      80.221387         19.661333     101.816872   
26         1.080954  

In [7]:
# Sort neatly by Country and Year
clean_df = research_df.sort_values(by=["Country", "Year"]).reset_index(drop=True)

# View the first 14 rows (Switzerland & United States from 2018 to 2024)
print(
    clean_df[[
        "Country",
        "Year",
        "Pubs_Per_100k",
        "Patents_Per_100k",
        "Research_Sub_Index",
    ]].head(14)
)

    Country  Year  Pubs_Per_100k  Patents_Per_100k  Research_Sub_Index
0     China  2018       0.180500          0.199643            0.013431
1     China  2019       0.431850          0.527349            0.146194
2     China  2020       1.036822          0.499219            0.230763
3     China  2021       3.262187          1.762067            0.764348
4     China  2022       2.694517          1.279540            0.346850
5     China  2023       2.196936          0.268047            0.235114
6     China  2024       2.997794          2.177985            0.699227
7   Germany  2018      29.393976          3.267470            1.974794
8   Germany  2019      30.339940         20.672241            6.002029
9   Germany  2020      60.403717         11.700538            9.330991
10  Germany  2021      37.700917          2.557367            3.749846
11  Germany  2022      29.226335          1.837480            2.510437
12  Germany  2023      21.400013         14.319753            2.167989
13  Ge

In [8]:
# Filter for South Africa and India, then sort neatly by Country and Year
target_countries_df = research_df[
    research_df["Country"].isin(["South Africa", "India"])
]
target_clean = target_countries_df.sort_values(
    by=["Country", "Year"]
).reset_index(drop=True)

# Display the main metrics for your Research Sub-Index
print(
    target_clean[[
        "Country",
        "Year",
        "Pubs_Per_100k",
        "Patents_Per_100k",
        "Research_Sub_Index",
    ]]
)

         Country  Year  Pubs_Per_100k  Patents_Per_100k  Research_Sub_Index
0          India  2018       1.788429          0.704429            0.242274
1          India  2019       3.616200          0.633343            0.453463
2          India  2020       1.410570          0.050155            0.132213
3          India  2021       2.145410          0.027534            0.138224
4          India  2022       2.076692          1.746090            0.317867
5          India  2023       3.830941          1.310116            0.616038
6          India  2024       1.466448          0.869834            0.285786
7   South Africa  2018      45.095000          1.825000            3.979098
8   South Africa  2019      21.059500         13.912200            2.814025
9   South Africa  2020       2.785650         16.774776            3.436976
10  South Africa  2021      42.004423         25.792603           11.590657
11  South Africa  2022      80.221387         19.661333            7.289412
12  South Af

In [15]:
import pandas as pd

BASE = "clean-pt2/"
PUB_FIELD = "All"    # replace with the AI field name if one exists
PAT_FIELD = "All"    # replace with the AI field name if one exists
YEARS = list(range(2018, 2025))

pubs = pd.read_csv(BASE + "publications_yearly_articles_6countries.csv")
pat = pd.read_csv(BASE + "patents_yearly_applications_6countries.csv")
pop = pd.read_csv(BASE + "population_6countries.csv")

print("Publication fields:", pubs["field"].unique())
print("Patent fields:", pat["field"].unique())

pubs = pubs[(pubs["field"] == PUB_FIELD) & pubs["year"].isin(YEARS)]
pat = pat[(pat["field"] == PAT_FIELD) & pat["year"].isin(YEARS)]

merged = (
    pop[["country", "year", "population"]]
    .merge(pubs[["country", "year", "num_articles"]], on=["country", "year"], how="left")
    .merge(pat[["country", "year", "num_patent_applications"]], on=["country", "year"], how="left")
)

out = merged.rename(columns={
    "num_articles": "ai_publications",
    "num_patent_applications": "ai_patents",
})[["country", "year", "ai_publications", "ai_patents", "population"]]

print("Rows:", len(out), "(expect 42)")
print("Missing values per column:")
print(out.isna().sum())

out.to_csv(BASE + "research_input.csv", index=False)
print("Saved to", BASE + "research_input.csv")

Publication fields: <StringArray>
[                  'AI Safety',                         'All',
       'Large language models',                    'Robotics',
             'Computer Vision', 'Natural Language Processing']
Length: 6, dtype: str
Patent fields: <StringArray>
[                              'All',          'Analytics and Algorithms',
               'Banking and Finance',            'Bio-Inspired Computing',
                          'Business',                   'Computer Vision',
           'Computing in Government',                   'Control Methods',
                    'Distributed AI',                         'Education',
                       'Fuzzy Logic',        'Industry and Manufacturing',
          'Knowledge Representation',       'Natural Language Processing',
        'Markov and Bayesian Models',             'Measuring and Testing',
                   'Neural Networks',              'Ontology Engineering',
             'Other and Unspecified',    'Personal 